# Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -qU transformers datasets optimum
!pip install -qU openai wandb
!pip install -qU json-repair faker
!pip install -qU vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 654, done.
remote: Counting objects: 100% (654/654), done.
remote: Compressing objects: 100% (492/492), done.
remote: Total 654 (delta 156), reused 423 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (654/654), 5.29 MiB | 21.16 MiB/s, done.
Resolving deltas: 100% (156/156), done.
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Imports


In [ ]:
from google.colab import userdata
import wandb
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests
import pandas as pd
from huggingface_hub import login

In [ ]:
wandb.login(key=userdata.get('wandb'))
login(token=userdata.get('HF_TOKEN'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc



Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [ ]:
df_training = pd.read_json('/content/drive/MyDrive/opinion-mining/train_labeled_v2.json', lines=True)
df_training.head()

,input,prediction
0,I charge it at night and skip taking the cord ...,"{""domain"":""electronics"",""aspects"":[{""term"":""Ba..."
1,I bought a HP Pavilion DV41222nr laptop and ha...,"{""domain"":""electronics"",""aspects"":[{""term"":""HP..."
2,The tech guy then said the service center does...,"{""domain"":""electronics"",""aspects"":[{""term"":""se..."
3,I investigated netbooks and saw the Toshiba NB...,"{""domain"":""electronics"",""aspects"":[{""term"":""Ne..."
4,The other day I had a presentation to do for a...,"{""domain"":""software"",""aspects"":[{""term"":""compu..."


# Formatting Finetune Data

In [ ]:
system_message = "\n".join(["You extract structured information from text and return only valid JSON.",
                            "",
                            "No explanation No introduction No conclusion"])

In [ ]:
llm_finetunning_data = []

for i, row in df_training.iterrows():
    input = row['input']
    output = row['prediction']

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": "\n".join(["Extract the domain and all aspect terms with their sentiment polarity.",
                                  "- Domains: electronics, restaurants, movies, books, software, general",
                                  "- Polarity: positive, negative, neutral",
                                  "- Extract ALL aspects mentioned in the text",
                                  "input :",
                                  input,
                                  "output format:",
                                  '{"domain":"...","aspects":[{"term":"...","polarity":"..."}, ...]}']),
        "input" : "",
        "output" : output,
        "history" : []
    })

random.Random(101).shuffle(llm_finetunning_data)


In [ ]:
len(llm_finetunning_data)

5959

In [ ]:
data_dir = "/content/drive/MyDrive/opinion-mining"

train_sample_sz = 5000

train_ds = llm_finetunning_data[:train_sample_sz]
eval_ds = llm_finetunning_data[train_sample_sz:]

os.makedirs(join(data_dir, "datasets", "llamafactory-finetune-data"), exist_ok=True)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [ ]:
join(data_dir, "datasets", "llamafactory-finetune-data", "val.json")

'/content/drive/MyDrive/opinion-mining/datasets/llamafactory-finetune-data/val.json'

# Llama-Factory Configuration

In [ ]:
# # Configure LLaMA-Factory for the new datasets

# # update /content/LLaMA-Factory/data/dataset_info.json and append
# ```
  "opinion_mining_finetune_train": {
        "file_name": "/content/drive/MyDrive/opinion-mining/datasets/llamafactory-finetune-data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "opinion_mining_finetune_val": {
        "file_name": "/content/drive/MyDrive/opinion-mining/datasets/llamafactory-finetune-data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
# ```

In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all

### dataset
dataset: opinion_mining_finetune_train
eval_dataset: opinion_mining_finetune_val
template: qwen
cutoff_len: 1000
##max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
resume_from_checkpoint: /content/drive/opinion-mining/models/checkpoint-1000
output_dir: /content/drive/MyDrive/opinion-mining/models/
logging_steps: 25
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 550
#disable_tqdm: true

report_to: wandb
run_name: opinion-mining-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "AbdallahSabry1/opinion-mining"
hub_private_repo: true
hub_strategy: checkpoint

Writing /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


# Finetuning

In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-04-23 10:55:16] llamafactory.hparams.parser:505 >> Process rank: 0, world